# 3. Grading from the volumetric features

Reproduces **Section 3.5** (the classifier panel), **Section 3.6** (the protocol) and
**Section 4.3** (the results):

| output | paper |
|---|---|
| six classifiers, accuracy, F1 and AUC, n = 369 | `tab:clf` |
| the panel extended with XGBoost, histogram gradient boosting and regularized logistic regression | Section 4.3 |
| SHAP attributions over the seventeen features | `fig:shap` |
| the label-shuffle control | Section 4, "shuffling the grade labels drops the random forest to an AUC of 0.43" |

Runs from `data_cache/` alone: no volumes, no models, a couple of minutes on a laptop.

## The protocol

Case-level five-fold stratified cross-validation, one row per case, scored by the area under the
ROC curve. The scaler sits inside the pipeline, so it is fit on training folds only. The same
protocol is used by every table in the paper.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src import config
print(config.describe())

In [ ]:
from src import cohort, evaluation, tables

predicted = cohort.extract_features()
grades = cohort.load_grade_labels()
X, y, cases = cohort.build_matrix(predicted, grades)
print(f"n = {len(cases)}  high-grade {int((y == 0).sum())}  low-grade {int(y.sum())}")
print(f"folds: {evaluation.CV}")

## Wiring proof against the superseded cohort

An earlier run of this pipeline lost one case to an unreadable copy of its file and silently
skipped it, which is why an older version of this table was on 368 cases. Reproducing that cohort
shows the protocol has not drifted; the corrected cohort of 369 is what the paper reports. Small
residual differences on the random forest come from library versions, not from the protocol.

In [ ]:
X_368, y_368, cases_368 = cohort.build_matrix(
    predicted, grades, exclude={cohort.PREVIOUSLY_SKIPPED})
superseded = {"SVM (RBF)": 0.937, "SVM (linear)": 0.919, "Random forest": 0.942,
              "Gradient boosting": 0.924, "Logistic reg.": 0.922, "$k$-NN ($k = 5$)": 0.879}
proof = tables.table_clf(X_368, y_368, include_xgboost=False)
display(tables.compare_to_paper(proof, "AUC", superseded, tolerance=0.005))

## tab:clf

Expected, on the 369-case cohort:

| Classifier | Accuracy | F1 | AUC |
|---|---|---|---|
| SVM (RBF) | 0.883 | 0.751 | **0.930** |
| SVM (linear) | 0.894 | 0.773 | 0.922 |
| Random forest | 0.913 | 0.804 | 0.929 |
| Gradient boosting | 0.903 | 0.737 | 0.919 |
| Logistic reg. | 0.902 | 0.782 | 0.920 |
| k-NN (k = 5) | 0.889 | 0.667 | 0.894 |

with XGBoost at 0.923 on the same features. Every AUC sits close to every other, which is the
point: the result is a property of the features, not of one lucky model.

In [ ]:
clf = tables.table_clf(X, y)
display(clf)

published = {"SVM (RBF)": 0.930, "SVM (linear)": 0.922, "Random forest": 0.929,
             "Gradient boosting": 0.919, "Logistic reg.": 0.920,
             "$k$-NN ($k = 5$)": 0.894, "XGBoost": 0.923}
display(tables.compare_to_paper(clf, "AUC", published))

## The extended panel

Adding XGBoost, histogram gradient boosting and $L_2$- and elastic-net-regularized logistic
regression does not raise the AUC. Widening the panel was a check against having picked a lucky
family of models, and it comes back negative.

In [ ]:
extended = tables.table_clf_extended(X, y)
display(extended.sort_values("AUC", ascending=False))
print(f"best AUC in the extended panel: {extended['AUC'].max():.3f}")

## Integrity control

With the grades permuted, the signal has to disappear. The random forest drops to **0.43**,
below chance, which is what a small cohort does when there is nothing left to learn.

In [ ]:
shuffled = evaluation.shuffled_label_control(X, y)
display(shuffled[["Accuracy", "F1", "AUC"]])
print(f"random forest on shuffled grades: {shuffled.loc['Random forest', 'AUC']:.3f}")

## fig:shap -- what carries the grade

The paper's figure was produced in the original exploratory notebook; this cell re-derives it
under the protocol above, with the explainer fit on the same random forest and the same seventeen
features. The reading it supports is that the strongest features are sub-region volumes and
fractions rather than texture or raw modality intensities, and in particular the enhancing and
necrotic measures, which is to say the compartments the method estimates without any label.

In [ ]:
# shap and matplotlib are needed only for this figure, so they are optional.
try:
    import matplotlib.pyplot as plt
    import shap
    HAVE_SHAP = True
except ImportError as exc:
    HAVE_SHAP = False
    print(f"{exc.name} is not installed; install shap and matplotlib to draw fig:shap")

In [ ]:
if HAVE_SHAP:
    from sklearn.preprocessing import StandardScaler

    from src import figures
    from src.subregions import FEATURE_NAMES

    scaler = StandardScaler().fit(X)
    forest = evaluation.CLASSIFIERS["Random forest"]().fit(scaler.transform(X), y)
    explanation = shap.TreeExplainer(forest).shap_values(scaler.transform(X))
    # A binary forest gives one matrix per class; the low-grade side is the positive class.
    values = explanation[..., 1] if np.ndim(explanation) == 3 else explanation

    order = np.argsort(np.abs(values).mean(0))[::-1]
    display(pd.Series(np.abs(values).mean(0)[order],
                      index=[FEATURE_NAMES[i] for i in order],
                      name="mean |SHAP|").to_frame().round(4))

In [ ]:
if HAVE_SHAP:
    figures.set_style()
    fig = plt.figure(figsize=(7.0, 4.6))
    shap.summary_plot(values, features=X, feature_names=FEATURE_NAMES, show=False,
                      color_bar_label="Feature value")
    plt.xlabel("SHAP value (impact on the low-grade decision)")
    plt.tight_layout()
    figures.save(fig, "shap_grade.png")
    plt.show()